# Database initialization

In [ ]:
from db_management_tools import DataLoader
dataset_dir = r"D:\AIC26"
data_loader = DataLoader(dataset_dir)

# Upload data

In [ ]:
data_loader.create_and_fill_database(
    "aic26_mini",
    upload_batch_size=1024,
    upload_num_workers=8
)

# Query initialization

In [1]:
from db_management_tools import QueryManager
from models import SigLIP2Model, BGEM3Model
from peripheral_tools import NotebookDisplay

image_text_feature_extractor = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")
text_text_feature_extractor = BGEM3Model(name="BAAI/bge-m3", device="cuda")

query_manager = QueryManager(
    image_text_feature_extractor=image_text_feature_extractor,
    text_text_feature_extractor=text_text_feature_extractor
)
display = NotebookDisplay(315, 5)

[INFO] Loading quantized google/siglip2-so400m-patch14-384
[INFO] google/siglip2-so400m-patch14-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# Query test

In [ ]:
query = input("Enter your query: ")
limit = int(input("Enter your search limit: "))
points = query_manager.search(
    "aic26_mini",
    query=query,
    search_limit=limit,
    using_vectors=["siglip2_dense"]
)

display.display(points)

# Benchmark

In [ ]:
from peripheral_tools import Benchmarker

test_settings = [
    {
        "using_vectors": ["siglip2_dense"]
    },
    {
        "using_vectors": ["siglip2_dense", "bgem3_sparse"]
    },
    {
        "using_vectors": ["siglip2_dense", "bgem3_dense"]
    },
    {
        "using_vectors": ["siglip2_dense", "bgem3_dense", "bgem3_sparse"]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.4,
        "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.4,
        "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.4,
        "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    },
]

benchmarker = Benchmarker(query_manager)
res = benchmarker.do_benchmark(
    "aic26_mini",
    test_questions=r"C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\question_bank\questions_with_answers.csv",
    test_settings=test_settings,
    tolerance=10,
    save_record=False
)

Doing benchmark: 100%|██████████| 33/33 [07:36<00:00, 13.83s/it]

Benchmark results:
           Result@1  Result@5  Result@10  Result@50  mean_MRR  median_MRR  min_MRR  max_MRR   std_MRR  mean_Runtime  median_Runtime  min_Runtime  max_Runtime  std_Runtime
Method 1   0.272727  0.545455   0.606061   0.757576  0.388911    0.250000      0.0      1.0  0.405598      0.031199        0.028512     0.020308     0.050072     0.008814
Method 2   0.212121  0.606061   0.757576   0.848485  0.397516    0.333333      0.0      1.0  0.354534      0.053535        0.052916     0.039525     0.073060     0.008668
Method 3   0.303030  0.606061   0.727273   0.787879  0.420179    0.250000      0.0      1.0  0.403373      0.053985        0.053948     0.041068     0.071466     0.007600
Method 4   0.303030  0.636364   0.757576   0.848485  0.438099    0.333333      0.0      1.0  0.394980      0.056568        0.056067     0.042432     0.081409     0.009899
Method 5   0.363636  0.666667   0.727273   0.848485  0.492014    0.333333      0.0      1.0  0.409494      2.374920        2.3